## SECTION 01: Database Selection and Schema Overview


In [ ]:
--show tables in the database
select * from sys.tables

# SECTION-02. Database Overview and Tables Used

AdventureWorks2022 is a sample OLTP database developed by Microsoft to simulate the operations of a bicycle manufacturing company. The database is organized into multiple schemas, each representing a different business domain. This project focuses on the core tables required for sales, customer, product, employee, and inventory analysis.

## Schemas Used

- **Sales**
- **Production**
- **Person**
- **HumanResources**
- **Purchasing**

## Core Tables

- `Sales.SalesOrderHeader`
- `Sales.SalesOrderDetail`
- `Sales.Customer`
- `Person.Person`
- `Production.Product`
- `Production.ProductCategory`
- `Production.ProductSubcategory`
- `Sales.SalesTerritory`
- `Sales.SalesPerson`
- `HumanResources.Employee`
- `Production.ProductInventory`



# SECTION-03. Business Problem Statement


## AIM

Adventure Works is a global bicycle manufacturing company that sells products through online and reseller channels across multiple regions. As the company's sales continue to grow, management requires data-driven insights to improve decision-making and business performance.

we will see few Business Questions below 

## Business Objectives

This project aims to answer the following business questions:

- How has sales performance changed over time?
- Which customers contribute the most revenue?
- Which products generate the highest sales and profit?
- Which sales territories perform the best?
- How effective are individual salespersons?
- Which products require inventory attention?
- What key performance indicators should management monitor?


The objectives outlined above represent the fundamental business analytics that every organization should perform to evaluate its performance. These analyses help identify strengths, uncover areas that need improvement, and provide valuable insights for making informed, data-driven decisions that support business growth and operational efficiency.

# SECTION-04. Sales Analysis

## Business Problem

Adventure Works management wants a comprehensive overview of the company's sales performance to understand revenue trends, sales growth, regional performance, salesperson contributions, and overall business health. As a Data Analyst, analyze the sales data and generate insights that answer the following:

- What is the total revenue generated by the company?
- What are the monthly and quarterly sales trends?
- What is the average order value?
- Which sales territories generate the highest revenue?
- Which salespersons contribute the most sales?
- What are the top 10 highest-value sales orders?



In [ ]:
--  Before running the below query, make sure to select the database in which you want to run the query.
--  sales.SalesOrderHeader
--  sales.SalesOrderDetail
--  sales.salesterritory
--  sales.salesperson
--  person.person


Commands completed successfully.

Total execution time: 00:00:00.001

In [11]:
set nocount on

--Querry.1
print 'Total Revenue:'
SELECT
    SUM(TotalDue) AS TotalRevenue
FROM Sales.SalesOrderHeader;
go

--Query.2
print 'Total Revenue by Year and Month:'
select top 10 year(OrderDate) as OrderYear, 
       month(OrderDate) as OrderMonth,
       datename(month, OrderDate) as MonthName,
       SUM(TotalDue) as TotalRevenue
 from Sales.SalesOrderHeader    
group by year(OrderDate), month(OrderDate), datename(month, OrderDate)

order by  OrderMonth,OrderYear
go

--query.3
print 'avg order value by year and month:'

select top 10 year(OrderDate) as OrderYear, 
       month(OrderDate) as OrderMonth,
       datename(month, OrderDate) as MonthName,
       AVG(TotalDue) as AvgOrderValue
 from Sales.SalesOrderHeader
 group by year(OrderDate), month(OrderDate), datename(month, OrderDate)
 order by  OrderMonth,OrderYear
 GO

 --querry.4
 print 'sales territory with highest revenue:'

go

WITH TerritoryRevenue AS
(
    SELECT
        t.TerritoryID,
        t.Name AS TerritoryName,
        SUM(s.TotalDue) AS TotalRevenue
    FROM Sales.SalesTerritory t
    INNER JOIN Sales.SalesOrderHeader s
        ON t.TerritoryID = s.TerritoryID
    GROUP BY
        t.TerritoryID,
        t.Name
)

SELECT
    TerritoryID,
    TerritoryName,
    TotalRevenue,
    DENSE_RANK() OVER (ORDER BY TotalRevenue DESC) AS RankID
FROM TerritoryRevenue
ORDER BY RankID;
go 

--querry.5
print 'Top 15 Salespersons by Revenue:';
with salespersonrevenue as 
(
    select s.salespersonid,
        p.FirstName + ' ' + p.LastName as SalesPersonName,
        SUM(s.TotalDue) as TotalRevenue
    from Sales.SalesOrderHeader s
    inner join person.person p
        on s.SalesPersonID = p.BusinessEntityID
    group by s.salespersonid, p.FirstName + ' ' +  p.LastName
)
select top 15 salespersonid, SalesPersonName, TotalRevenue
    from salespersonrevenue
    order by TotalRevenue desc
GO


--querry.6
print 'Top 10 Sales Orders by Total Due:';

select top (10)
    soh.salesorderid,
    soh.orderdate,
    p.firstname + ' ' + p.lastname as customer_name,
    sp.firstname + ' ' + sp.lastname as salesperson_name,
    st.name as territory,
    soh.totaldue
from sales.salesorderheader soh

inner join sales.customer c
    on soh.customerid = c.customerid

left join person.person p
    on c.personid = p.businessentityid

left join sales.salesperson s
    on soh.salespersonid = s.businessentityid

left join person.person sp
    on s.businessentityid = sp.businessentityid

left join sales.salesterritory st
    on soh.territoryid = st.territoryid

order by soh.totaldue desc;
go



Total Revenue:

TotalRevenue  
--------------
123216786.1159
(1 row)

Total Revenue by Year and Month:

OrderYear | OrderMonth | MonthName | TotalRevenue
----------+------------+-----------+-------------
2023      | 1          | January   | 2325568.5984
2024      | 1          | January   | 2340061.5521
2025      | 1          | January   | 4783231.6666
2023      | 2          | February  | 1620826.1021
2024      | 2          | February  | 2629322.4479
2025      | 2          | February  | 3987943.6249
2023      | 3          | March     | 3336347.4716
2024      | 3          | March     | 3826046.0691
2025      | 3          | March     | 5585675.8075
2023      | 4          | April     | 1871923.5039
(10 rows)

avg order value by year and month:

OrderYear | OrderMonth | MonthName | AvgOrderValue
----------+------------+-----------+--------------
2023      | 1          | January   | 8979.0293    
2024      | 1          | January   | 5850.1538    
2025      | 1          | January   | 2239.340